Visual Storyteller - inference

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import os

In [12]:
# load trained model
checkpoint = torch.load('model.pth', map_location='cpu')
vocab = checkpoint['vocab']
embed_dim = checkpoint['embed_dim']
hidden_dim = checkpoint['hidden_dim']
vocab_size = len(vocab)

# convert ids back to words
idx_to_word = {v: k for k, v in vocab.items()}
print(f"vocab size: {vocab_size}")

vocab size: 5241


In [8]:
class EncoderCNN(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        resnet = models.resnet18(weights='DEFAULT')
        #remove final classification layer
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)
        #freeze encoder weights
        for param in self.resnet.parameters():
            param.requires_grad = False
        self.fc = nn.Linear(512, embed_dim)

    def forward(self, imgs):
        with torch.no_grad():
            features = self.resnet(imgs)
        features = features.squeeze(-1).squeeze(-1)
        return self.fc(features)

class DecoderLSTM(nn.Module):
    def __init__(self, embed_dim, hidden_dim, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.init_hidden = nn.Linear(embed_dim, hidden_dim)


    def forward(self, features, captions):
        #image features as initial hidden state
        hidden = self.init_hidden(features).unsqueeze(0)
        cell = torch.zeros_like(hidden)
        embeddings = self.embedding(captions[:, :-1])
        outputs, _ = self.lstm(embeddings, (hidden, cell))
        return self.fc(outputs)    

In [9]:
# rebuild model and load weights
encoder = EncoderCNN(embed_dim=embed_dim)
decoder = DecoderLSTM(embed_dim=embed_dim, hidden_dim=hidden_dim, vocab_size=vocab_size)
encoder.load_state_dict(checkpoint['encoder'])
decoder.load_state_dict(checkpoint['decoder'])

encoder.eval()
decoder.eval()


print(f"model loaded successfully")

model loaded successfully


In [14]:
def generate_caption(image_path: str, model: any) -> str:
    # image preprocessing.same transforms as training
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    img = Image.open(image_path).convert('RGB')
    img = transform(img).unsqueeze(0)


    encoder_model, decoder_model = model
    encoder_model.eval()
    decoder_model.eval()


    with torch.no_grad():
        features = encoder_model(img)
        hidden = decoder_model.init_hidden(features).unsqueeze(0)
        cell = torch.zeros_like(hidden)

        # start with <START> token
        word_idx = torch.tensor([[vocab['<START>']]])
        caption = []

        for _ in range(30):
            embeddings = decoder_model.embedding(word_idx)
            output, (hidden, cell) = decoder_model.lstm(embeddings, (hidden, cell))
            prediction = decoder_model.fc(output.squeeze(1))
            word_idx = prediction.argmax(1).unsqueeze(1)
            word = idx_to_word[word_idx.item()]

            if word == '<END>':
                break
            if word not in ['<PAD>', '<UNK>', '<START>']:
                caption.append(word)
    return ' '.join(caption)

In [17]:
# checking 10 examples
import random
all_images = os.listdir("Images")
sample = random.sample(all_images, 10)

for img_name in sample:
    img_path = os.path.join("Images", img_name)
    caption = generate_caption(img_path, (encoder, decoder))
    print(f"{img_name}: {caption}")

2900274587_f2cbca4c58.jpg: a person is standing in a river with a fishing net .
84713990_d3f3cef78b.jpg: a group of people are rowing a canoe in a lake .
1388346434_524d0b6dfa.jpg: a man in a black shirt and jeans is standing in front of a brick wall .
2593695271_4d9cc9bd6f.jpg: a man in a white shirt and a woman in a black dress and a woman in a white dress .
2851304910_b5721199bc.jpg: a man in a black shirt and jeans is standing on a sidewalk with a red and black umbrella .
3143991972_7193381aeb.jpg: a man and a woman are sitting on a couch .
2693539377_5442430f81.jpg: a man and a woman are sitting on a bench in a park .
2860202109_97b2b22652.jpg: a man in a black shirt and a woman in a black shirt .
430803349_a66c91f64e.jpg: a black dog is swimming in a lake .
3033257301_e2c8a39b04.jpg: a group of people are standing in a line of a large , green tent .


In [18]:
# good,average and failure examples
examples = [
    ("Images/84713990_d3f3cef78b.jpg", "good"),
    ("Images/3033257301_e2c8a39b04.jpg", "average"),
    ("Images/2851304910_b5721199bc.jpg", "failure"),
]

for path, label in examples:
    caption = generate_caption(path, (encoder, decoder))
    print(f"[{label}] {path}")
    print(f"caption: {caption}")
    print()

[good] Images/84713990_d3f3cef78b.jpg
caption: a group of people are rowing a canoe in a lake .

[average] Images/3033257301_e2c8a39b04.jpg
caption: a group of people are standing in a line of a large , green tent .

[failure] Images/2851304910_b5721199bc.jpg
caption: a man in a black shirt and jeans is standing on a sidewalk with a red and black umbrella .



my overall impression is that model pick up general scene patterns but struggles with specific details and sometimes invents content that isn't there.